# Engram: embed exported chunks on Kaggle

Use this when a job is too slow to embed on the laptop GPU.

1. On the Sources page, upload or sync with **Embed later with the Kaggle notebook** ticked. The job stops at `awaiting embedding`.
2. From `backend/`, export its chunks: `uv run python -m app.cli export-chunks --job <job id> --out chunks.jsonl`
3. Add `chunks.jsonl` to a **private** Kaggle dataset and attach it to this notebook.
4. Turn on a GPU accelerator (T4), turn on internet access, and run all cells. The model comes from the Hugging Face Hub at the revision in the export's manifest.
5. Download `manifest.json` and every `vectors-*.parquet` file from `/kaggle/working` into one folder.
6. From `backend/`, publish them: `uv run python -m app.cli import-vectors --job <job id> --in <folder>`

If a session ends early, save its output as a dataset, attach it along with the export, and run the notebook again. Shards that are already finished are copied instead of encoded again.

In [ ]:
# Shard writing. This cell needs no GPU, so the backend tests run it as it is.
import json
import shutil
from pathlib import Path

import pyarrow as pa
import pyarrow.parquet as pq


def read_export(chunks_path):
    """The manifest and the chunks from an export-chunks file."""
    with open(chunks_path, encoding="utf-8") as file:
        manifest = json.loads(file.readline())["manifest"]
        chunks = [json.loads(line) for line in file if line.strip()]
    if len(chunks) != manifest["chunk_count"]:
        raise ValueError(
            f"the export lists {manifest['chunk_count']} chunks but holds {len(chunks)}"
        )
    return manifest, chunks


def shard_name(number):
    return f"vectors-{number:05d}.parquet"


def finished_shard(name, ids, done_dirs):
    """A shard with exactly these chunk ids from this or an earlier session, or None."""
    for folder in done_dirs:
        path = Path(folder) / name
        if not path.exists():
            continue
        try:
            if pq.read_table(path, columns=["chunk_id"]).column("chunk_id").to_pylist() == ids:
                return path
        except Exception:  # a shard cut short when a session stopped is encoded again
            pass
    return None


def run(chunks_path, out_dir, encode, shard_size, done_dirs):
    """Encode the export in shards of shard_size chunks and write them to out_dir.

    encode takes a list of texts and returns one vector per text. A shard already in one of
    done_dirs with the same chunk ids is reused, so a new session picks up where the last
    one stopped. manifest.json is copied from the export for import-vectors to check.
    """
    manifest, chunks = read_export(chunks_path)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    encoded = reused = 0
    names = []
    for number, start in enumerate(range(0, len(chunks), shard_size)):
        batch = chunks[start : start + shard_size]
        ids = [chunk["chunk_id"] for chunk in batch]
        name = shard_name(number)
        names.append(name)
        done = finished_shard(name, ids, done_dirs)
        if done is not None:
            if done.resolve() != (out_dir / name).resolve():
                shutil.copyfile(done, out_dir / name)
            reused += 1
            continue
        vectors = [[float(value) for value in vector] for vector in encode([c["text"] for c in batch])]
        if len(vectors) != len(batch) or any(len(v) != manifest["dimension"] for v in vectors):
            raise ValueError(
                f"encode must return one vector of length {manifest['dimension']} per text"
            )
        table = pa.table(
            {"chunk_id": ids, "vector": pa.array(vectors, type=pa.list_(pa.float32()))}
        )
        partial = out_dir / (name + ".partial")
        pq.write_table(table, partial)
        partial.replace(out_dir / name)
        encoded += 1
    # Shards left from a longer export would be read by import-vectors, so remove them.
    for leftover in out_dir.glob("vectors-*.parquet"):
        if leftover.name not in names:
            leftover.unlink()
    (out_dir / "manifest.json").write_text(json.dumps(manifest), encoding="utf-8")
    return {"chunks": len(chunks), "shards_encoded": encoded, "shards_reused": reused}

In [ ]:
# Kaggle: load BGE-M3 at the export's pinned revision on the GPU and write the shards.
import os

import torch
from sentence_transformers import SentenceTransformer

INPUT = Path("/kaggle/input")
WORKING = Path("/kaggle/working")
SHARD_SIZE = 1000
BATCH_SIZE = 16

exports = sorted(INPUT.rglob("chunks*.jsonl"))
if len(exports) != 1:
    raise SystemExit(f"Attach exactly one export-chunks file. Found: {[str(p) for p in exports]}")
chunks_path = exports[0]
manifest, _ = read_export(chunks_path)
print(f"{chunks_path}: {manifest['chunk_count']} chunks for {manifest['model_id']}")

# The pinned revision has only pytorch_model.bin. Skip the safetensors conversion download.
os.environ.setdefault("DISABLE_SAFETENSORS_CONVERSION", "1")
model = SentenceTransformer(
    manifest["model_id"],
    revision=manifest["model_revision"],
    device="cuda",
    model_kwargs={"dtype": torch.float16},
)
model.max_seq_length = 1024  # the same settings as the backend's BgeM3Embedder
encode_document = getattr(model, "encode_document", model.encode)


def encode(texts):
    return encode_document(
        texts,
        batch_size=BATCH_SIZE,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )


# /kaggle/working starts empty in a new session, so earlier output attached as an input
# counts as finished too.
done_dirs = [WORKING, *sorted({path.parent for path in INPUT.rglob("vectors-*.parquet")})]
summary = run(chunks_path, WORKING, encode, shard_size=SHARD_SIZE, done_dirs=done_dirs)
print(summary)
print("Download manifest.json and vectors-*.parquet from /kaggle/working, then run import-vectors.")